# 02 — The Agent Class: Deep Dive

## What You'll Learn
- Every field on the `Agent` class and what it controls
- How Runner manages the agent lifecycle
- Sessions, multi-turn conversations, and state persistence
- Models, instructions (system prompts), and `output_key`

> **Prerequisites:** Notebook 01 (installation & first agent)


## 2.1 The Agent Class — Complete Reference

`Agent` is the central abstraction in ADK. Every agent is defined by:

```python
Agent(
    name='...',              # Required: unique identifier
    model='...',             # Required: which LLM to use
    instruction='...',       # The system prompt (behavior definition)
    description='...',       # For multi-agent discovery
    tools=[...],             # Tools the agent can use
    sub_agents=[...],        # Child agents this agent can delegate to
    output_key='...',        # Auto-save response to session.state
    # + callbacks, code_executor, planner, etc. (covered later)
)
```

### The `name` Field
- Must be unique within an agent hierarchy
- Used in event.author to identify who produced what
- Used in multi-agent routing (`transfer_to_agent`)

### The `model` Field
- String like `'gemini-2.5-flash'` or a `BaseLlm` instance
- LiteLLM format supported: `'openrouter:provider/model'`
- The model MUST support function calling (all Gemini 2 models do)

### The `instruction` Field
- This is the **system prompt** — it defines the agent's entire behavior
- Can be a string OR a callable (dynamic instructions)
- **Best practice:** Include ROLE, TOOLS, CONSTRAINTS, OUTPUT sections


In [2]:
# ─── Core ADK imports for every project ────────────────────────────
# ADK 2.x uses a clean import structure. Here are the four essentials:
import asyncio  # Required: session creation is async in ADK 2.x
from google.adk import Agent  # The core agent class — your building block
from google.adk.runners import Runner  # Executes agents and manages sessions
from google.adk.sessions.in_memory_session_service import InMemorySessionService  # Stores conversation state
from google.genai.types import Content, Part  # Message types (NOTE: google.genai.types, NOT google.adk.types!)
print("✅ Core imports ready")
# ─── Load API key from .env file ───────────────────────────────────
# Google ADK needs GOOGLE_API_KEY to call Gemini models.
# The .env file (in this directory) stores your key safely.
# This cell reads it and sets the environment variable for the session.
import os
from dotenv import load_dotenv, find_dotenv

# Load the .env file
load_dotenv(find_dotenv())
has_key = bool(os.environ.get("GOOGLE_API_KEY", ""))
print(f'API key: {"✅ LOADED" if has_key else "❌ NOT FOUND — create .env file"}')


✅ Core imports ready
API key: ✅ LOADED


## 2.2 Instructions — The Art of Prompting Agents

The instruction is the most important field. A well-written instruction
is the difference between a useful agent and a frustrating one.

### Good vs Bad Instructions

```python
# BAD: Vague, no constraints
instruction = "Be helpful"

# GOOD: Role, tools, constraints, output format
instruction = """
ROLE: You are a precise research assistant.
TOOLS: Use google_search for current data. Always cite sources.
CONSTRAINTS: Never fabricate facts. Keep answers under 3 paragraphs.
OUTPUT: End with "Sources:" and list URLs.
"""
```

### Instruction Sections to Include
1. **ROLE**: Who the agent is ("You are a...")
2. **TOOLS**: When and how to use each tool
3. **CONSTRAINTS**: Hard rules ("never do X", "always do Y")
4. **OUTPUT**: Expected format ("Return as bullet points", "Use markdown")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CREATING AGENTS WITH GOOD INSTRUCTIONS
# ═══════════════════════════════════════════════════════════════════

# Example 1: A simple agent with a minimal instruction
simple_agent = Agent(
    name="simple_bot",
    model="gemini-2.5-flash",
    instruction="You are a helpful assistant. Be concise.",
)
print(f"Created: {simple_agent.name} | model={simple_agent.model}")

# Example 2: A production-quality instruction
# This pattern is what you should use for real agents
PRODUCTION_INSTRUCTION = """
ROLE: You are a technical documentation assistant for a software company.
TOOLS: None currently — answer from your training knowledge.
CONSTRAINTS:
  - Never make up API endpoints or code that doesn't exist
  - If unsure, say "I don't know" and suggest where to look
  - Keep answers under 4 paragraphs
OUTPUT: Use markdown formatting. Code in ``` blocks with language tags.
"""

pro_agent = Agent(
    name="docs_assistant",
    model="gemini-2.5-flash",
    instruction=PRODUCTION_INSTRUCTION,
    description="Helps with technical documentation questions",
)
print(f"Created: {pro_agent.name} with detailed instruction")


## 2.3 Runner & Sessions — How Execution Works

### The Runner Class
`Runner` is the execution engine. It:
1. Takes an `Agent` and a `SessionService`
2. Manages the agent loop (send to LLM → get response → execute tools → repeat)
3. Yields `Event` objects for streaming
4. Handles errors and retries

### Sessions: The Agent's Memory
A **session** is a container that holds:
- **Conversation history** (all messages exchanged)
- **State** (a dict for storing data across turns)
- **Artifacts** (files created during the conversation)

Sessions are identified by `(app_name, user_id, session_id)`.
This triplet uniquely identifies one conversation.

### Session Lifecycle
```
create_session() → [turn 1] → [turn 2] → ... → session persists
```
Each call to `runner.run()` with the same session_id continues
the conversation — the LLM sees all previous messages.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# RUNNER AND SESSION DEMO — Single turn with event inspection
# ═══════════════════════════════════════════════════════════════════

async def demo_runner():
    # ─── Create an agent ─────────────────────────────────────────
    agent = Agent(
        name="inspector",
        model="gemini-2.5-flash",
        instruction="You are a helpful bot. Answer questions clearly.",
    )

    # ─── Create session (ASYNC — must await) ─────────────────────
    svc = InMemorySessionService()
    session = await svc.create_session(
        app_name="tutorial", user_id="user1", session_id="runner_demo"
    )

    # ─── Create runner ───────────────────────────────────────────
    runner = Runner(agent=agent, app_name="tutorial", session_service=svc)

    # ─── Build message ───────────────────────────────────────────
    msg = Content(role="user", parts=[Part.from_text(text="What is your name?")])

    # ─── Run and inspect every event ─────────────────────────────
    print("Streaming events from runner.run():\n")
    for event in runner.run(user_id="user1", session_id=session.id, new_message=msg):
        # Each event tells us WHO produced it and WHAT was produced
        if event.content and event.content.parts and event.author != "user":
            for part in event.content.parts:
                if part.text:
                    # part.text = the actual text content
                    print(f"  [{event.author}] TEXT: {part.text[:200]}")
                if part.function_call:
                    # part.function_call = LLM wants to invoke a tool
                    print(f"  [{event.author}] TOOL CALL: {part.function_call.name}")
                if part.function_response:
                    # part.function_response = result from tool execution
                    print(f"  [{event.author}] TOOL RESULT: {part.function_response.name}")

await demo_runner()


## 2.4 Multi-Turn Conversations

One of ADK's key features: **conversations persist across turns**.
When you call `runner.run()` again with the same `session_id`,
the agent sees all previous messages in that session.

This is fundamentally different from raw LLM APIs where you must
manually track and resend the entire conversation history.


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MULTI-TURN CONVERSATION — Agent remembers across calls
# ═══════════════════════════════════════════════════════════════════

async def multi_turn():
    # ─── Create a single agent + session ─────────────────────────
    agent = Agent(
        name="memory_bot",
        model="gemini-2.5-flash",
        instruction="You are a helpful assistant. Remember user facts.",
    )
    svc = InMemorySessionService()
    s = await svc.create_session(app_name="t", user_id="u", session_id="mt")
    r = Runner(agent=agent, app_name="t", session_service=svc)

    # ─── Turn 1: Tell the agent something ────────────────────────
    m1 = Content(role="user", parts=[Part.from_text(
        text="My name is Alice and I live in Berlin."
    )])
    print("Turn 1 — User: My name is Alice and I live in Berlin.\n")
    for e in r.run(user_id="u", session_id=s.id, new_message=m1):
        if e.content and e.content.parts:
            for p in e.content.parts:
                if p.text and not p.thought:
                    print(f"  Agent: {p.text}")

    # ─── Turn 2: Ask about the remembered fact ───────────────────
    # SAME session_id → agent sees Turn 1's messages!
    m2 = Content(role="user", parts=[Part.from_text(
        text="What's my name and where do I live?"
    )])
    print("\nTurn 2 — User: What's my name and where do I live?\n")
    for e in r.run(user_id="u", session_id=s.id, new_message=m2):
        if e.content and e.content.parts:
            for p in e.content.parts:
                if p.text and not p.thought:
                    print(f"  Agent: {p.text}")

await multi_turn()


## 2.5 `output_key` — Saving Agent Responses to State

`output_key` tells ADK: "Save this agent's final text response into
`session.state[key]`." This is the primary way to pass data between
agents in a pipeline.

### How It Works
1. Agent runs and produces a text response
2. ADK automatically saves that text to `session.state[output_key]`
3. Downstream agents or code can read `session.state[output_key]`

### Use Cases
- **Pipeline data passing**: Agent A extracts → Agent B transforms
- **Caching**: Avoid re-running expensive operations
- **Inspection**: See what each agent produced


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# OUTPUT_KEY DEMO — Auto-saving agent responses to session.state
# ═══════════════════════════════════════════════════════════════════

async def output_key_demo():
    # Agent configured to save its response under "keywords"
    extractor = Agent(
        name="extractor",
        model="gemini-2.5-flash",
        instruction="Extract exactly 3 keywords from the text. Return comma-separated ONLY.",
        output_key="keywords",  # ← THIS is the magic: auto-saves to session.state["keywords"]
    )

    svc = InMemorySessionService()
    s = await svc.create_session(app_name="t", user_id="u", session_id="ok")
    r = Runner(agent=extractor, app_name="t", session_service=svc)

    # Send text for extraction
    m = Content(role="user", parts=[Part.from_text(
        text="Machine learning models can hallucinate when lacking diverse training data."
    )])

    # Run the agent (we don't care about streaming here)
    for _ in r.run(user_id="u", session_id=s.id, new_message=m):
        pass  # Consume all events to complete execution

    # ─── Read the saved output from session state ────────────────
    # get_session() is ALSO async in ADK 2.x!
    sess = await svc.get_session(app_name="t", user_id="u", session_id="ok")
    print("Session state keys:", list(sess.state.keys()))
    print("Keywords extracted:", sess.state.get("keywords", "NOT SET"))
    
    # session.state is a regular Python dict — you can read/write freely
    # This is how pipelines pass data between stages!

await output_key_demo()


## Summary

| Concept | What it does |
|---|---|
| `Agent.name` | Unique ID for routing and event tracking |
| `Agent.model` | LLM to use (Gemini, OpenAI, Anthropic, Ollama...) |
| `Agent.instruction` | System prompt — defines behavior, constraints, output |
| `Agent.output_key` | Auto-saves response to session.state |
| `Runner` | Executes agent loop, yields Events |
| `Session` | Stores conversation history + state (async create/get) |
| `Event` | One step in the agent loop (text, tool call, error) |

**Next:** [03_tools.ipynb](./03_tools.ipynb) — adding capabilities with tools.
